# Continuum: which retrieval model should we use?

Compares embedding models and rerankers on **LongMemEval**, a benchmark of 500 questions
asked over long user–assistant chat histories. Every chat turn is treated as one "memory",
and each question searches only its own history, like Continuum's per-user search.

**How to run on Kaggle**
1. Settings → Accelerator: **GPU T4 x2** (or P100). Settings → Internet: **On**.
2. Run All. For a quick first pass, set `MAX_QUESTIONS = 100` in the config cell.
3. Copy the tables printed at the end (or download `longmemeval_results.csv` from the Output tab).

The decision we want: does any model that fits on the laptop (next to Qwen on a 6 GB GPU)
clearly beat `bge-small-en-v1.5`, which Continuum uses today?

In [ ]:
import importlib.util
import os
import subprocess
import sys

# Installs only what's missing on Kaggle; skipped during local smoke tests.
# einops is only needed by nomic-embed-text.
if os.environ.get("LME_SMOKE_TEST") != "1":
    for package, module in [("rank_bm25", "rank_bm25"), ("sentence-transformers", "sentence_transformers"), ("einops", "einops")]:
        if importlib.util.find_spec(module) is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [ ]:
import csv
import gc
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import hf_hub_download
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

# Set the environment variable LME_SMOKE_TEST=1 for a tiny local check instead of the benchmark.
SMOKE_TEST = os.environ.get("LME_SMOKE_TEST") == "1"

DATASET_REPO = "xiaowu0162/longmemeval-cleaned"
DATASET_FILE = "longmemeval_s_cleaned.json"  # ~277 MB, roughly 40-50 sessions per question
MAX_QUESTIONS = None  # None = all 500; e.g. 100 for a quick first run
INDEX_ROLES = ("user", "assistant")  # chat turns that become searchable "memories"
MAX_SEQ_LENGTH = 256  # tokens of each turn the embedding models read
BATCH_SIZE = 128
RECALL_CUTOFFS = (1, 5, 10)
SESSION_CUTOFF = 5

PRODUCTION_MODEL = "bge-small-en-v1.5"  # what Continuum uses today
RERANK_CANDIDATES = 20  # production model's top results that each reranker reorders

BGE_QUERY = "Represent this sentence for searching relevant passages: "

# name: (Hugging Face id, query prefix, document prefix, trust_remote_code)
EMBEDDING_MODELS = {
    "bge-small-en-v1.5": ("BAAI/bge-small-en-v1.5", BGE_QUERY, "", False),
    "bge-base-en-v1.5": ("BAAI/bge-base-en-v1.5", BGE_QUERY, "", False),
    "bge-large-en-v1.5": ("BAAI/bge-large-en-v1.5", BGE_QUERY, "", False),
    "e5-small-v2": ("intfloat/e5-small-v2", "query: ", "passage: ", False),
    "e5-base-v2": ("intfloat/e5-base-v2", "query: ", "passage: ", False),
    "gte-small": ("thenlper/gte-small", "", "", False),
    "gte-base": ("thenlper/gte-base", "", "", False),
    "arctic-embed-s": ("Snowflake/snowflake-arctic-embed-s", BGE_QUERY, "", False),
    "arctic-embed-m-v1.5": ("Snowflake/snowflake-arctic-embed-m-v1.5", BGE_QUERY, "", False),
    "nomic-embed-text-v1.5": ("nomic-ai/nomic-embed-text-v1.5", "search_query: ", "search_document: ", True),
    "all-MiniLM-L6-v2": ("sentence-transformers/all-MiniLM-L6-v2", "", "", False),
}

RERANKERS = {
    "bge-reranker-base": "BAAI/bge-reranker-base",
    "bge-reranker-v2-m3": "BAAI/bge-reranker-v2-m3",
    "ms-marco-MiniLM-L6-v2": "cross-encoder/ms-marco-MiniLM-L6-v2",
}

if SMOKE_TEST:
    DATASET_FILE = "longmemeval_oracle.json"
    MAX_QUESTIONS = 80
    EMBEDDING_MODELS = {name: EMBEDDING_MODELS[name] for name in (PRODUCTION_MODEL, "all-MiniLM-L6-v2")}
    RERANKERS = {"bge-reranker-base": RERANKERS["bge-reranker-base"]}

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))

## Load LongMemEval
Turns marked `has_answer` are the evidence for a question. Questions whose id ends in `_abs`
are **abstention** questions: the history doesn't contain the answer, so the right response is
"I don't know". They're excluded from ranking metrics and used to test whether a model's
scores can tell answerable questions from unanswerable ones.

In [ ]:
dataset_path = hf_hub_download(DATASET_REPO, DATASET_FILE, repo_type="dataset")
instances = json.loads(Path(dataset_path).read_text(encoding="utf-8"))

if MAX_QUESTIONS:
    instances = random.Random(0).sample(instances, min(MAX_QUESTIONS, len(instances)))

texts: list[str] = []  # every distinct turn, embedded once per model
text_ids: dict[str, int] = {}
questions = []

for instance in instances:
    units, unit_sessions, relevant = [], [], set()

    for session_id, session in zip(instance["haystack_session_ids"], instance["haystack_sessions"]):
        for turn in session:
            content = turn["content"].strip()
            if turn["role"] not in INDEX_ROLES or not content:
                continue

            if content not in text_ids:
                text_ids[content] = len(texts)
                texts.append(content)

            if turn.get("has_answer"):
                relevant.add(len(units))

            units.append(text_ids[content])
            unit_sessions.append(session_id)

    questions.append({
        "id": instance["question_id"],
        "type": instance["question_type"],
        "question": instance["question"],
        "abstention": instance["question_id"].endswith("_abs"),
        "units": np.array(units, dtype=np.int64),
        "sessions": unit_sessions,
        "relevant": relevant,
        "answer_sessions": set(instance["answer_session_ids"]),
    })

question_index = {question["id"]: i for i, question in enumerate(questions)}
answerable = [q for q in questions if not q["abstention"] and q["relevant"]]
abstention = [q for q in questions if q["abstention"] and len(q["units"])]
skipped = len(questions) - len(answerable) - len(abstention)

print(f"{len(questions)} questions: {len(answerable)} answerable, {len(abstention)} abstention"
      + (f", {skipped} skipped (no evidence turn in INDEX_ROLES)" if skipped else ""))
print(f"{len(texts):,} distinct turns to embed; "
      f"median {np.median([len(q['units']) for q in questions]):.0f} turns searched per question")

## Metrics
- **MRR** and **R@k** use the same definitions as Continuum's local harness: R@k is
  relevant turns in the top k divided by min(k, relevant turns).
- **session R@5**: are the conversations that contain the answer among the top 5 sessions?
- **abstention AUROC**: the chance that an answerable question's top score beats an
  unanswerable one's. 0.5 means a score threshold can't tell them apart; 1.0 means perfectly.

In [ ]:
results = {}  # method -> summary metrics
reciprocal_ranks = {}  # method -> {question id: 1 / rank of first relevant turn}
type_mrr = {}  # method -> {question type: MRR}


def first_relevant_rank(order, relevant):
    for rank, position in enumerate(order, start=1):
        if int(position) in relevant:
            return rank
    return None


def recall_at_k(order, relevant, k):
    return len({int(p) for p in order[:k]} & relevant) / min(k, len(relevant))


def session_recall_at_k(order, question, k):
    top_sessions = []
    for position in order:
        session = question["sessions"][int(position)]
        if session not in top_sessions:
            top_sessions.append(session)
            if len(top_sessions) == k:
                break
    return len(set(top_sessions) & question["answer_sessions"]) / min(k, len(question["answer_sessions"]))


def auroc(positives, negatives):
    if not positives or not negatives:
        return float("nan")
    p = np.asarray(positives)[:, None]
    n = np.asarray(negatives)[None, :]
    return float((p > n).mean() + 0.5 * (p == n).mean())


def evaluate(name, rank, extra=None):
    """Score a retrieval method. `rank(question)` returns (turn positions best-first, top score)."""
    rows, top_answerable, top_abstention = [], [], []

    for question in answerable:
        order, top_score = rank(question)
        found = first_relevant_rank(order, question["relevant"])
        row = {"id": question["id"], "type": question["type"], "rr": 0.0 if found is None else 1 / found}
        for k in RECALL_CUTOFFS:
            row[f"R@{k}"] = recall_at_k(order, question["relevant"], k)
        row[f"session R@{SESSION_CUTOFF}"] = session_recall_at_k(order, question, SESSION_CUTOFF)
        rows.append(row)
        top_answerable.append(top_score)

    for question in abstention:
        top_abstention.append(rank(question)[1])

    summary = {"MRR": float(np.mean([row["rr"] for row in rows]))}
    for key in [f"R@{k}" for k in RECALL_CUTOFFS] + [f"session R@{SESSION_CUTOFF}"]:
        summary[key] = float(np.mean([row[key] for row in rows]))
    summary["abstention AUROC"] = auroc(top_answerable, top_abstention)
    summary.update(extra or {})

    results[name] = summary
    reciprocal_ranks[name] = {row["id"]: row["rr"] for row in rows}
    type_mrr[name] = {
        question_type: float(np.mean([row["rr"] for row in rows if row["type"] == question_type]))
        for question_type in sorted({row["type"] for row in rows})
    }

    print(f"{name:40s} MRR {summary['MRR']:.3f} | R@1 {summary['R@1']:.3f} | "
          f"R@5 {summary['R@5']:.3f} | abstention AUROC {summary['abstention AUROC']:.3f}")


def free_gpu_memory():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## BM25 baseline (keyword matching, same tokenizer as Continuum)

In [ ]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


tokens = [tokenize(text) for text in texts]


def bm25_rank(question):
    index = BM25Okapi([tokens[t] for t in question["units"]])
    scores = index.get_scores(tokenize(question["question"]))
    order = np.argsort(-scores, kind="stable")
    return order, float(scores[order[0]])


evaluate("bm25", bm25_rank)

## Embedding models
A model that fails to load is skipped, and the rest still run.

In [ ]:
first_stage_orders = {}  # question id -> production model's top candidates, for reranking

for name, (model_id, query_prefix, document_prefix, remote_code) in EMBEDDING_MODELS.items():
    try:
        model = SentenceTransformer(model_id, device=DEVICE, trust_remote_code=remote_code)
        model.max_seq_length = MAX_SEQ_LENGTH
        if DEVICE == "cuda":
            model.half()

        started = time.perf_counter()
        document_vectors = model.encode(
            [document_prefix + text for text in texts],
            batch_size=BATCH_SIZE, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True,
        ).astype(np.float32)
        turns_per_second = len(texts) / (time.perf_counter() - started)

        query_vectors = model.encode(
            [query_prefix + question["question"] for question in questions],
            batch_size=BATCH_SIZE, normalize_embeddings=True, convert_to_numpy=True,
        ).astype(np.float32)
        params = sum(p.numel() for p in model.parameters()) / 1e6
    except Exception as error:
        print(f"{name}: skipped ({type(error).__name__}: {error})")
        free_gpu_memory()
        continue

    del model
    free_gpu_memory()

    def embedding_rank(question, document_vectors=document_vectors, query_vectors=query_vectors):
        scores = document_vectors[question["units"]] @ query_vectors[question_index[question["id"]]]
        order = np.argsort(-scores, kind="stable")
        return order, float(scores[order[0]])

    evaluate(name, embedding_rank, {"params (M)": params, "turns/s": turns_per_second})

    if name == PRODUCTION_MODEL:
        for question in answerable + abstention:
            first_stage_orders[question["id"]] = embedding_rank(question)[0][:RERANK_CANDIDATES]

## Rerankers
Each reranker reorders the production model's top 20, the two-stage setup Continuum would use.

In [ ]:
if not first_stage_orders:
    print(f"{PRODUCTION_MODEL} didn't run, so rerankers are skipped.")
    RERANKERS = {}

pool_questions = answerable + abstention
pairs, spans = [], {}
for question in pool_questions:
    pool = first_stage_orders.get(question["id"], [])
    spans[question["id"]] = (len(pairs), len(pairs) + len(pool))
    pairs.extend((question["question"], texts[question["units"][int(p)]]) for p in pool)

for name, model_id in RERANKERS.items():
    try:
        reranker = CrossEncoder(model_id, device=DEVICE, max_length=512)
        started = time.perf_counter()
        pair_scores = np.asarray(reranker.predict(pairs, batch_size=64, show_progress_bar=True), dtype=np.float32)
        pairs_per_second = len(pairs) / (time.perf_counter() - started)
        params = sum(p.numel() for p in getattr(reranker, "model", reranker).parameters()) / 1e6
    except Exception as error:
        print(f"{name}: skipped ({type(error).__name__}: {error})")
        free_gpu_memory()
        continue

    del reranker
    free_gpu_memory()

    def rerank(question, pair_scores=pair_scores):
        start, end = spans[question["id"]]
        pool = first_stage_orders[question["id"]]
        local_order = np.argsort(-pair_scores[start:end], kind="stable")
        return pool[local_order], float(pair_scores[start:end][local_order[0]])

    evaluate(f"{PRODUCTION_MODEL} + {name}", rerank, {"params (M)": params, "pairs/s": pairs_per_second})

## Results

In [ ]:
def format_table(headers, rows):
    widths = [max(len(str(row[i])) for row in [headers, *rows]) for i in range(len(headers))]
    render = lambda row: "  ".join(
        str(cell).ljust(width) if i == 0 else str(cell).rjust(width)
        for i, (cell, width) in enumerate(zip(row, widths))
    )
    return "\n".join([render(headers), "  ".join("-" * width for width in widths), *map(render, rows)])


def fmt(value, digits=3):
    return "-" if value is None else f"{value:.{digits}f}"


baseline = reciprocal_ranks.get(PRODUCTION_MODEL)
metric_columns = ["MRR", "R@1", "R@5", "R@10", f"session R@{SESSION_CUTOFF}", "abstention AUROC"]
ordered = sorted(results, key=lambda name: -results[name]["MRR"])

summary_rows = []
for name in ordered:
    summary = results[name]
    if baseline is None or name == PRODUCTION_MODEL:
        versus = "baseline" if name == PRODUCTION_MODEL else "-"
    else:
        wins = sum(rr > baseline[qid] for qid, rr in reciprocal_ranks[name].items())
        losses = sum(rr < baseline[qid] for qid, rr in reciprocal_ranks[name].items())
        versus = f"+{wins} / -{losses}"
    summary_rows.append(
        [name, *(fmt(summary.get(column)) for column in metric_columns),
         fmt(summary.get("params (M)"), 0), versus]
    )

print(f"LongMemEval ({DATASET_FILE}): {len(answerable)} answerable + {len(abstention)} abstention questions, "
      f"max {MAX_SEQ_LENGTH} tokens per turn\n")
print(format_table(["method", *metric_columns, "params (M)", f"vs {PRODUCTION_MODEL}"], summary_rows))
print(f"\n'vs {PRODUCTION_MODEL}': questions where the first relevant turn ranked higher (+) or lower (-).")

question_types = sorted({t for per_type in type_mrr.values() for t in per_type})
print("\nMRR by question type")
print(format_table(
    ["method", *question_types],
    [[name, *(fmt(type_mrr[name].get(t)) for t in question_types)] for name in ordered],
))

with open(OUTPUT_DIR / "longmemeval_results.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["method", *metric_columns, "params (M)", f"vs {PRODUCTION_MODEL}", *question_types])
    for name, row in zip(ordered, summary_rows):
        writer.writerow([*row, *(fmt(type_mrr[name].get(t)) for t in question_types)])

print(f"\nSaved {OUTPUT_DIR / 'longmemeval_results.csv'}")

## How to read this
- **Decision rule:** switch from `bge-small-en-v1.5` only if a model of about **350M parameters
  or fewer** (it has to fit beside Qwen on a 6 GB GPU) beats it clearly: roughly **+0.03 MRR or
  more, with far more wins than losses**. Then confirm it on Continuum's local harness before
  changing production.
- **Abstention AUROC near 0.5** means similarity scores can't detect "the answer isn't here",
  so a score threshold won't fix abstention. That needs a relevance check by the LLM instead.
- LongMemEval turns are much longer than Continuum's one-line memories (median about 550
  characters), so absolute numbers won't transfer; the ranking between models is what matters.